# 🧠 EmoWave — SVM Pipeline (Google Colab)

**Tái hiện bài báo Wang et al. (2014):**  
Power Spectrum + Asymmetry Features + LDS Smoothing + SVM

---

## 📋 Hướng dẫn chuẩn bị

1. **Upload data DEAP lên Google Drive:**
   - Tạo thư mục `EmoWave/data/deap/` trong Google Drive
   - Upload các file `s01.dat` → `s32.dat` vào đó
   - Cấu trúc: `My Drive/EmoWave/data/deap/s01.dat`

2. **Runtime:** Chọn `Runtime → Change runtime type → T4 GPU` (tùy chọn, CPU cũng đủ)

3. **Chạy lần lượt từng cell** từ trên xuống

## 1. Mount Google Drive & Cấu hình

In [8]:
from google.colab import drive
drive.mount('/content/drive')

# ─── CẤU HÌNH ───────────────────────────────────────────
# Đường dẫn tới thư mục chứa s01.dat → s32.dat trên Drive
DATA_DIR = "/content/drive/MyDrive/EmoWave/data/deap"

# Số subjects muốn dùng (1-32)
# Dùng ít hơn để chạy nhanh hơn. 32 = full dataset.
N_SUBJECTS = 32

# Bật/tắt các tính năng
USE_LDS = True           # LDS smoothing
USE_GRIDSEARCH = True    # GridSearch (tắt để chạy nhanh)

print("✓ Google Drive mounted!")
print(f"Data dir: {DATA_DIR}")

ModuleNotFoundError: No module named 'google'

In [ ]:
# Kiểm tra data có tồn tại không
import os
files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.dat')])
print(f"Tìm thấy {len(files)} files:")
for f in files:
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1024 / 1024
    print(f"  {f}  ({size_mb:.1f} MB)")

## 2. Import Libraries

In [3]:
import numpy as np
import pickle
import os
import json
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)

print("✓ All imports OK")

✓ All imports OK


## 3. DEAP Loader (tự chứa, không cần import từ project)

In [4]:
# ─── Cấu hình DEAP ─────────────────────────────────────
N_TRIALS = 40
N_CHANNELS = 32
SFREQ = 128
EPOCH_SEC = 1
SAMPLES_PER_EPOCH = SFREQ * EPOCH_SEC  # 128
BASELINE_SAMPLES = 3 * SFREQ  # 384


def load_subject(subject_id, data_dir=DATA_DIR):
    """Đọc 1 file .dat"""
    filename = os.path.join(data_dir, f"s{subject_id:02d}.dat")
    with open(filename, "rb") as f:
        return pickle.load(f, encoding="latin1")


def make_labels_2class(labels_raw):
    return (labels_raw[:, 0] >= 5).astype(int)


def make_labels_4class(labels_raw):
    valence = labels_raw[:, 0]
    arousal = labels_raw[:, 1]
    classes = np.zeros(len(valence), dtype=int)
    classes[(valence >= 5) & (arousal >= 5)] = 0
    classes[(valence <  5) & (arousal >= 5)] = 1
    classes[(valence <  5) & (arousal <  5)] = 2
    classes[(valence >= 5) & (arousal <  5)] = 3
    return classes


def segment_trial(trial_data, baseline=BASELINE_SAMPLES, epoch_len=SAMPLES_PER_EPOCH):
    signal = trial_data[:, baseline:]
    n_epochs = signal.shape[1] // epoch_len
    return np.array([signal[:, i*epoch_len:(i+1)*epoch_len] for i in range(n_epochs)])


def load_all_subjects(data_dir=DATA_DIR, n_subjects=N_SUBJECTS, label_type="2class"):
    X_list, y_list, g_list = [], [], []
    for sid in range(1, n_subjects + 1):
        print(f"  Loading subject {sid:02d}/{n_subjects}...", end="\r")
        try:
            subject = load_subject(sid, data_dir)
        except FileNotFoundError:
            print(f"  [WARN] s{sid:02d}.dat not found — skipping")
            continue

        data_raw = subject["data"][:, :N_CHANNELS, :]
        labels_raw = subject["labels"]

        if label_type == "2class":
            trial_labels = make_labels_2class(labels_raw)
        else:
            trial_labels = make_labels_4class(labels_raw)

        for t in range(N_TRIALS):
            epochs = segment_trial(data_raw[t])
            n_ep = len(epochs)
            X_list.append(epochs)
            y_list.append(np.full(n_ep, trial_labels[t]))
            g_list.append(np.full(n_ep, sid))

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0).astype(np.int64)
    g = np.concatenate(g_list, axis=0).astype(np.int64)

    print(f"  ✓ Xong! X: {X.shape}, y: {y.shape}")
    print(f"  Phân bố nhãn: { {int(k): int(v) for k, v in zip(*np.unique(y, return_counts=True))} }")
    return X, y, g

print("✓ DEAP Loader ready")

NameError: name 'DATA_DIR' is not defined

## 4. Feature Extraction (Power Spectrum + Asymmetry)

In [ ]:
FREQ_BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45),
}

ASYMMETRY_PAIRS = [
    (0, 16), (1, 17), (2, 18), (3, 19), (4, 20), (5, 21), (6, 22),
    (7, 23), (8, 24), (9, 25), (10, 26), (11, 27), (12, 28), (13, 29),
]


def extract_power_spectrum(epoch, sfreq=SFREQ):
    fft_vals = np.abs(np.fft.rfft(epoch, axis=-1)) ** 2
    freqs = np.fft.rfftfreq(epoch.shape[-1], 1.0 / sfreq)
    band_powers = []
    for (f_low, f_high) in FREQ_BANDS.values():
        idx = np.where((freqs >= f_low) & (freqs <= f_high))[0]
        if len(idx) == 0:
            band_powers.append(np.zeros(epoch.shape[0]))
        else:
            power = np.log1p(fft_vals[:, idx].mean(axis=-1))
            band_powers.append(power)
    return np.stack(band_powers, axis=-1)


def extract_asymmetry(power_features):
    asymmetry = []
    for left_idx, right_idx in ASYMMETRY_PAIRS:
        diff = power_features[left_idx] - power_features[right_idx]
        asymmetry.append(diff)
    return np.array(asymmetry)


def extract_all_features(epoch, sfreq=SFREQ):
    power = extract_power_spectrum(epoch, sfreq)
    asym = extract_asymmetry(power)
    return np.concatenate([power.flatten(), asym.flatten()])


def extract_features_dataset(X_epochs, sfreq=SFREQ):
    print("  Trích xuất features...")
    start = time.time()
    features = []
    for i in range(len(X_epochs)):
        features.append(extract_all_features(X_epochs[i], sfreq))
        if (i + 1) % 10000 == 0:
            elapsed = time.time() - start
            speed = (i+1) / elapsed
            eta = (len(X_epochs) - i - 1) / speed
            print(f"    {i+1}/{len(X_epochs)}  ({speed:.0f} epochs/s, ETA: {eta:.0f}s)")
    result = np.array(features, dtype=np.float32)
    print(f"  ✓ Feature shape: {result.shape} ({time.time()-start:.1f}s)")
    return result

print("✓ Feature extractors ready")

## 5. LDS Smoothing

In [ ]:
def lds_smoothing(features_seq, alpha=0.3):
    smoothed = np.zeros_like(features_seq)
    smoothed[0] = features_seq[0]
    for t in range(1, len(features_seq)):
        smoothed[t] = alpha * features_seq[t] + (1 - alpha) * smoothed[t - 1]
    return smoothed


def apply_lds(X_features, epochs_per_trial=60, alpha=0.3):
    print(f"  Áp dụng LDS smoothing (alpha={alpha})...")
    X_smoothed = np.copy(X_features)
    n_trials = len(X_features) // epochs_per_trial
    for t in range(n_trials):
        s, e = t * epochs_per_trial, (t + 1) * epochs_per_trial
        X_smoothed[s:e] = lds_smoothing(X_features[s:e], alpha)
    print("  ✓ LDS xong!")
    return X_smoothed

print("✓ LDS Smoothing ready")

## 6. SVM Training Functions

In [ ]:
def train_svm_simple(X_train, y_train, kernel="linear", C=1.0):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel=kernel, C=C, random_state=42))
    ])
    pipeline.fit(X_train, y_train)
    return pipeline


def train_svm_gridsearch(X_train, y_train):
    param_grid = [
        {'svm__kernel': ['linear'], 'svm__C': [0.01, 0.1, 1, 10, 100]},
        {'svm__kernel': ['rbf'], 'svm__C': [0.1, 1, 10, 100],
         'svm__gamma': ['scale', 'auto', 0.001, 0.01]},
    ]

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(random_state=42))
    ])

    grid = GridSearchCV(
        pipeline, param_grid,
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        scoring='accuracy',
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    print("  Đang tìm hyperparameters tốt nhất...")
    start = time.time()
    grid.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"  ✓ GridSearch trong {elapsed:.1f}s")
    print(f"  Best params: {grid.best_params_}")
    print(f"  Best CV accuracy: {grid.best_score_*100:.2f}%")

    return grid.best_estimator_, grid.best_params_, grid.best_score_

print("✓ SVM functions ready")

## 7. Evaluation Functions

In [ ]:
RESULTS_DIR = "/content/results"

def evaluate_and_save(model, X_test, y_test, label_type="2class",
                      best_params=None, cv_score=None):
    os.makedirs(RESULTS_DIR, exist_ok=True)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    names = (["Negative", "Positive"] if label_type == "2class"
             else ["Vui vẻ", "Sợ hãi", "Buồn", "Thư giãn"])

    print(f"\n  {'═'*40}")
    print(f"  SVM — {label_type}")
    print(f"  {'═'*40}")
    print(f"  Accuracy:      {acc*100:.2f}%")
    print(f"  F1 (weighted): {f1*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=names))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Dự đoán', fontsize=12)
    ax.set_ylabel('Thực tế', fontsize=12)
    ax.set_title(f'SVM Confusion Matrix ({label_type})\n'
                 f'Accuracy: {acc*100:.2f}%', fontsize=14)
    plt.tight_layout()

    fig_path = os.path.join(RESULTS_DIR, f"confusion_matrix_svm_{label_type}.png")
    plt.savefig(fig_path, dpi=150)
    plt.show()

    # JSON
    results = {
        "model": "SVM",
        "label_type": label_type,
        "accuracy": round(acc, 4),
        "f1_weighted": round(f1, 4),
        "best_params": str(best_params) if best_params else None,
        "cv_score": round(cv_score, 4) if cv_score else None,
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_test, y_pred, target_names=names, output_dict=True
        ),
    }

    json_path = os.path.join(RESULTS_DIR, f"svm_results_{label_type}.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"  → Saved: {fig_path}")
    print(f"  → Saved: {json_path}")
    return results

print("✓ Evaluation functions ready")

---
## 🚀 8. CHẠY PIPELINE CHÍNH

Cell dưới đây sẽ chạy toàn bộ pipeline cho cả **2 lớp** và **4 lớp**.

In [ ]:
all_results = {}

for label_type in ["2class", "4class"]:
    print(f"\n{'='*60}")
    print(f"  EmoWave SVM Pipeline — {label_type}")
    print(f"{'='*60}")

    # 1. Load data
    print("\n[1] Loading DEAP data...")
    X, y, groups = load_all_subjects(label_type=label_type)

    # 2. Extract features
    print("\n[2] Feature extraction (Power Spectrum + Asymmetry)...")
    X_feat = extract_features_dataset(X)

    # 3. LDS smoothing
    if USE_LDS:
        print("\n[3] LDS smoothing...")
        X_feat = apply_lds(X_feat, epochs_per_trial=60, alpha=0.3)
    else:
        print("\n[3] LDS smoothing — SKIPPED")

    # 4. Train/test split
    print("\n[4] Train/test split (80/20, stratified)...")
    X_train, X_test, y_train, y_test = train_test_split(
        X_feat, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"  Train: {X_train.shape}, Test: {X_test.shape}")

    # 5. Train SVM
    if USE_GRIDSEARCH:
        print("\n[5] GridSearchCV (Linear + RBF)...")
        model, best_params, cv_score = train_svm_gridsearch(X_train, y_train)
    else:
        print("\n[5] Training SVM (linear, C=1.0)...")
        model = train_svm_simple(X_train, y_train, kernel='linear', C=1.0)
        best_params, cv_score = None, None

    # 6. Evaluate
    print("\n[6] Evaluation...")
    results = evaluate_and_save(
        model, X_test, y_test, label_type, best_params, cv_score
    )
    all_results[label_type] = results

# ── Tổng kết ──
print(f"\n{'='*60}")
print("  ✓ SVM Pipeline hoàn tất!")
print(f"{'='*60}")
print(f"\n  {'Label Type':<12} {'Accuracy':>10} {'F1':>10}")
print(f"  {'─'*12} {'─'*10} {'─'*10}")
for lt, r in all_results.items():
    print(f"  {lt:<12} {r['accuracy']*100:>9.2f}% {r['f1_weighted']*100:>9.2f}%")

## 9. Download kết quả về máy local

In [ ]:
# Copy kết quả sang Google Drive để giữ lại
import shutil

drive_results = "/content/drive/MyDrive/EmoWave/results"
os.makedirs(drive_results, exist_ok=True)

for f in os.listdir(RESULTS_DIR):
    src = os.path.join(RESULTS_DIR, f)
    dst = os.path.join(drive_results, f)
    shutil.copy2(src, dst)
    print(f"  Copied: {f}")

print(f"\n✓ Kết quả đã lưu vào Google Drive: {drive_results}")
print("  Bạn có thể copy các file này về thư mục results/ trong project local.")

In [ ]:
# Hoặc download trực tiếp từ Colab
from google.colab import files

for f in os.listdir(RESULTS_DIR):
    filepath = os.path.join(RESULTS_DIR, f)
    files.download(filepath)
    print(f"  Downloaded: {f}")